**Single-Image Super Resolution**
- Folder name can be set according to your need.
- In this part, you need to complete the Gaussian kernel function **get_kernel()**.
- Your output file "result/zebra_test_single.png" should be similar to "reference/zebra_test_single_golden.png" **(PSNR>60dB)**.

In [ ]:
from pathlib import Path
import os


def find_hw03_root():
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / 'optimization-based').exists() and (path / 'convnet-based').exists():
            return Path(os.path.relpath(path, cwd))

        hw03 = path / 'hw03'
        if (hw03 / 'optimization-based').exists() and (hw03 / 'convnet-based').exists():
            return Path(os.path.relpath(hw03, cwd))

    raise FileNotFoundError('Run this notebook from the repository root, hw03, or an hw03 subfolder.')


FOLDER_NAME = str(find_hw03_root())


=====Environment setting=====

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    'imageio': 'imageio==2.36.0',
    'skimage': 'scikit-image==0.24.0',
    'proximal': 'proximal',
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])


In [ ]:
import scipy.misc
from scipy.datasets import ascent
scipy.misc.ascent = ascent

from proximal.utils.utils import *
from proximal.halide.halide import *
from proximal.lin_ops import *
from proximal.prox_fns import *
from proximal.algorithms import *
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
import cv2
import numpy as np
import imageio.v2 as imageio
import time

 ===== Gaussian kernel function =====
 - Complete the following function, and make sure that the PSNR in the last part is higher than **60 dB**.

In [ ]:
def get_kernel(gau_N=11, gau_std=1.2):
    '''
    Compute the 2D gaussian kernel

    Inputs:
        gau_N:    gaussian kernel size
        gau_std:  standard deviation of the gaussian kernel
    Outputs:
        kernel:   gaussian kernel with the shape (N, N)
    '''
    # ===== write your kernel here ===== #

    coords = np.arange(-(gau_N // 2), gau_N // 2 + 1)
    yy, xx = np.meshgrid(coords, coords)
    cy, cx = -1.5, -1.5

    kernel = np.exp(-((xx - cx) * (xx - cx) + (yy - cy) * (yy - cy)) / (2.0 * gau_std * gau_std))
    kernel /= np.sum(kernel)

    # ================================== #

    return kernel

===== Soving for Single-Image Super Resolution =====
- Do not modify the remaining code.

In [ ]:
def solve(img, kernel, lamb):
    img4x_size = (4*img.shape[0], 4*img.shape[1], img.shape[2])
    img4x = cv2.resize(img, dsize=(img4x_size[1], img4x_size[0]), interpolation=cv2.INTER_CUBIC)
    tstart = time.time()
    x = Variable(img4x_size)
    # formulate problem
    prob = Problem(norm1(subsample(conv(kernel, x, dims=2), (4, 4, 1)) - img) + lamb*group_norm1(grad(x, dims=2), [3]))
    # solve problem
    result = prob.solve(verbose=True, solver='pc', x0=img4x, max_iters=1000)
    img_solved = x.value
    t_int = time.time() - tstart
    print('Elapsed time: {} seconds'.format(t_int))

    return img_solved


In [ ]:
# set parameters
gaussian_N = 13
gaussian_std = 1.2
lamb = 1e-2

# read tset image
img_test_path = f'{FOLDER_NAME}/optimization-based/image/LR_zebra_test.png'
# img_test_path = f'{FOLDER_NAME}/optimization-based/own_image/LR_test.png'

img_test = imageio.imread(img_test_path)/255.0
# solve the optimization problem
gau_kernel = get_kernel(gau_N=gaussian_N, gau_std=gaussian_std)
gau_rgb = np.repeat(np.expand_dims(gau_kernel, axis=2), repeats=3, axis=2)
img_solved = solve(img_test, gau_rgb, lamb)

# save result image
img_out = np.round(255*np.clip(img_solved, 0.0, 1.0)).astype('uint8')
imageio.imwrite(f'{FOLDER_NAME}/optimization-based/result/zebra_test_single.png', img_out)
# imageio.imwrite(f'{FOLDER_NAME}/optimization-based/result/own_test_single.png', img_out)


===== Testing on Single-Image Super Resolution =====
- Make sure that the PSNR with respect to TAs reference golden image is higher than **60 dB**.

In [ ]:
def check_ans(img_urs_path, img_ref_path):
    img_urs = imageio.imread(img_urs_path)
    img_ref = imageio.imread(img_ref_path)
    psnr = psnr_metric(img_ref, img_urs)
    if np.isinf(psnr):
        print("===> PSNR: inf dB (images are identical)")
    else:
        print("===> PSNR: {:.4f} dB".format(psnr))
    ssim = ssim_metric(img_ref, img_urs, channel_axis=-1, data_range=255)
    print('===> SSIM: {:.4f}'.format(ssim))

In [ ]:
# check answer
img_urs_path = f'{FOLDER_NAME}/optimization-based/result/zebra_test_single.png'
# img_urs_path = f'{FOLDER_NAME}/optimization-based/result/own_test_single.png'

img_ref_path =  f'{FOLDER_NAME}/optimization-based/reference/HR_zebra_test.png'
# img_ref_path =  f'{FOLDER_NAME}/optimization-based/own_image/HR_test.png'

print('===== compare with original HR image ===== ')
check_ans(img_urs_path, img_ref_path)

img_ref_path =  f'{FOLDER_NAME}/optimization-based/reference/zebra_test_single_golden.png'
print('\n===== compare with TAs reference answer ===== ')
check_ans(img_urs_path, img_ref_path)